# 💻 Client 1: Single Real Epoch Fine-Tuning on Kaggle GPU

### 📌 Overview & Setup Instructions
- **Project**: Real Federated Medical AI System (Single Real Communication Round)
- **Role**: **Client 1** (Assigned to Friend's PC: Ryzen 7, RTX 4050, 16GB RAM)
- **Partition Assignment**: **Dirichlet Partition Index 0** (`client_0.csv`: 1,366 samples, 57.0% positive)
- **Training Configuration**: **1 Local Epoch** using Weighted BCE (`pos_weight = 3.2596`)
- **Initial Global Weights**: Step 11 FedProx ($\\mu = 0.1$) checkpoint (`best_fedprox_model_mu_0.1.pt`) or Step 5 Baseline.
- **Output Artifacts**: `client1_weights.pt` and `client1_metadata.json`
- **Hardware Requirement**: **Kaggle GPU T4 x2**



In [ ]:
# Cell 1: Environment Setup & Hardware Disclosure
!pip install -q pydicom torchvision scikit-learn matplotlib pandas numpy opencv-python

import os
import sys
import time
import json
import copy
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pydicom

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix

print("=== SYSTEM & HARDWARE DISCLOSURE ===")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available?   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU Device Name   : {gpu_name}")
    
    try:
        test_tensor = torch.zeros(1).cuda()
        print(f"GPU Tensor Check  : SUCCESS (Compute capability supported on {gpu_name})")
        device = torch.device("cuda")
    except Exception as e:
        print()
        print("!" * 80)
        print("CRITICAL GPU COMPATIBILITY ERROR DETECTED:")
        print(f"  {e}")
        print("REASON: Kaggle Tesla P100 (compute capability sm_60) is incompatible with modern PyTorch builds.")
        print("ACTION REQUIRED: In Kaggle's right-hand panel, under Settings -> Accelerator:")
        print("                 Switch accelerator from 'GPU P100' to 'GPU T4 x2'.")
        print("!" * 80)
        print()
        raise RuntimeError("Incompatible GPU (Tesla P100). Please switch Kaggle Accelerator setting to 'GPU T4 x2'.")
else:
    print("WARNING: CUDA Not Available! Please attach GPU accelerator in Kaggle Settings.")
    device = torch.device("cpu")

OUTPUT_DIR = Path("/kaggle/working/outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
HEATMAPS_DIR = OUTPUT_DIR / "heatmaps"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HEATMAPS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output Directory  : {OUTPUT_DIR}")

def get_paths(dataset_name="federated-medical-ai-outputs"):
    """
    Centralized path resolution engine. Checks /kaggle/input/[dataset_name]/ FIRST for pre-saved
    checkpoints, partition files, or outputs before assuming a step needs retraining.
    """
    input_base = Path(f"/kaggle/input/{dataset_name}")
    if input_base.exists():
        print(f"[OK] Discovered attached dataset at: {input_base}")
        return {
            "output_dir": OUTPUT_DIR,
            "checkpoint_dir": input_base / "checkpoints" if (input_base / "checkpoints").exists() else CHECKPOINT_DIR,
            "partition_dir": input_base / "client_partitions" if (input_base / "client_partitions").exists() else OUTPUT_DIR / "client_partitions",
            "is_attached": True
        }

    input_dirs = list(Path("/kaggle/input").glob("**/checkpoints"))
    if len(input_dirs) > 0:
        matched_dir = input_dirs[0]
        matched_parent = matched_dir.parent
        ds_name = matched_parent.parts[3] if len(matched_parent.parts) > 3 else "attached-dataset"
        print(f"[OK] Discovered attached dataset containing checkpoints at: /kaggle/input/{ds_name}/checkpoints/")
        return {
            "output_dir": OUTPUT_DIR,
            "checkpoint_dir": matched_dir,
            "partition_dir": matched_parent / "client_partitions" if (matched_parent / "client_partitions").exists() else OUTPUT_DIR / "client_partitions",
            "is_attached": True
        }

    return {
        "output_dir": OUTPUT_DIR,
        "checkpoint_dir": CHECKPOINT_DIR,
        "partition_dir": OUTPUT_DIR / "client_partitions",
        "is_attached": False
    }

def find_checkpoint(filename, dataset_name="federated-medical-ai-outputs"):
    """
    Checks /kaggle/input/ attached datasets FIRST before assuming a checkpoint needs retraining.
    Returns the resolved Path object.
    """
    working_path = CHECKPOINT_DIR / filename
    if working_path.exists():
        print(f"[CACHE HIT] Found checkpoint in local session working dir: {working_path}")
        return working_path

    input_matches = list(Path("/kaggle/input").glob(f"**/{filename}"))
    if len(input_matches) > 0:
        found_path = input_matches[0]
        ds_name = found_path.parts[3] if len(found_path.parts) > 3 else dataset_name
        print(f"[OK] [CACHE HIT] Found pre-saved checkpoint in attached Kaggle dataset!")
        print(f"     Loaded from: /kaggle/input/{ds_name}/checkpoints/{filename}")
        print(f"     To reuse in future sessions: Add Input -> search for {ds_name} -> Add, then load from /kaggle/input/{ds_name}/checkpoints/{filename}")
        print("     >> Reusing prior verified model weights without retraining! <<")
        return found_path

    print(f"[INFO] Checkpoint '{filename}' not found in /kaggle/input/ attached datasets or local working dir.")
    return working_path

def find_client_partitions():
    """
    Checks /kaggle/input/ attached datasets FIRST for 5-client Dirichlet partitions before regenerating.
    """
    working_dir = OUTPUT_DIR / "client_partitions"
    if working_dir.exists() and len(list(working_dir.glob("client_*.csv"))) == 5:
        print(f"[CACHE HIT] Found 5 client partition files in local working dir: {working_dir}")
        return working_dir

    input_matches = list(Path("/kaggle/input").glob("**/client_partitions"))
    for match in input_matches:
        if len(list(match.glob("client_*.csv"))) == 5:
            ds_name = match.parts[3] if len(match.parts) > 3 else "attached-dataset"
            print(f"[OK] [CACHE HIT] Found pre-saved client partitions in attached dataset!")
            print(f"     Loaded from: /kaggle/input/{ds_name}/client_partitions/")
            return match

    return working_dir



In [ ]:
# Cell 2: RSNA Data Mount Verification (Strict Error Check)
RSNA_DATA_DIR = Path("/kaggle/input/rsna-pneumonia-detection-challenge")
LABELS_CSV = RSNA_DATA_DIR / "stage_2_train_labels.csv"

if not LABELS_CSV.exists():
    alt_paths = list(Path("/kaggle/input").glob("**/stage_2_train_labels.csv"))
    if len(alt_paths) > 0:
        LABELS_CSV = alt_paths[0]
        RSNA_DATA_DIR = LABELS_CSV.parent
        print(f"[OK] Found RSNA Labels CSV at: {LABELS_CSV}")
    else:
        raise FileNotFoundError(
            f"CRITICAL ERROR: RSNA Dataset not found at {RSNA_DATA_DIR}! "
            "Please add dataset 'rsna-pneumonia-detection-challenge' to this Kaggle notebook before running."
        )

IMAGES_DIR = RSNA_DATA_DIR / "stage_2_train_images"
if not IMAGES_DIR.exists():
    alt_imgs = list(RSNA_DATA_DIR.glob("**/stage_2_train_images"))
    if len(alt_imgs) > 0:
        IMAGES_DIR = alt_imgs[0]

print(f"[OK] RSNA Labels CSV : {LABELS_CSV}")
print(f"[OK] RSNA Images Dir : {IMAGES_DIR}")



In [ ]:
# Cell 3: DICOM PyTorch Dataset & Splitter
def parse_and_split_rsna(labels_csv_path, subset_size=6000, seed=42):
    df_raw = pd.read_csv(labels_csv_path)
    grouped = []
    for pid, group in df_raw.groupby("patientId"):
        target = group["Target"].iloc[0]
        grouped.append({"patientId": pid, "Target": int(target)})
    df_unique = pd.DataFrame(grouped)

    if subset_size and len(df_unique) > subset_size:
        df_unique = df_unique.sample(n=subset_size, random_state=seed).reset_index(drop=True)

    shuffled = df_unique.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_total = len(shuffled)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)

    train_df = shuffled.iloc[:n_train].reset_index(drop=True)
    val_df = shuffled.iloc[n_train:n_train + n_val].reset_index(drop=True)
    test_df = shuffled.iloc[n_train + n_val:].reset_index(drop=True)

    print(f"Data Split Summary (Subset Size = {len(df_unique)} Patients):")
    print(f"  - Train : {len(train_df)} patients ({train_df['Target'].mean()*100:.2f}% positive)")
    print(f"  - Val   : {len(val_df)} patients ({val_df['Target'].mean()*100:.2f}% positive)")
    print(f"  - Test  : {len(test_df)} patients ({test_df['Target'].mean()*100:.2f}% positive)")
    return train_df, val_df, test_df

class RSNADICOMDataset(Dataset):
    def __init__(self, df, images_dir, image_size=(224, 224)):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = row["patientId"]
        target = int(row.get("Target", 0))

        dcm_path = self.images_dir / f"{pid}.dcm"
        if not dcm_path.exists():
            png_path = self.images_dir / f"{pid}.png"
            if png_path.exists():
                img = Image.open(png_path).convert("RGB")
            else:
                raise FileNotFoundError(f"DICOM image not found for patient {pid} at {dcm_path}")
        else:
            dcm = pydicom.dcmread(str(dcm_path))
            arr = dcm.pixel_array.astype(np.float32)
            arr_min, arr_max = arr.min(), arr.max()
            if arr_max > arr_min:
                arr = (arr - arr_min) / (arr_max - arr_min) * 255.0
            else:
                arr = np.zeros_like(arr)
            img = Image.fromarray(arr.astype(np.uint8)).convert("RGB")

        tensor = self.transform(img)
        return tensor, torch.tensor(target, dtype=torch.float32)

train_df, val_df, test_df = parse_and_split_rsna(LABELS_CSV, subset_size=6000, seed=42)

train_dataset = RSNADICOMDataset(train_df, IMAGES_DIR)
val_dataset = RSNADICOMDataset(val_df, IMAGES_DIR)
test_dataset = RSNADICOMDataset(test_df, IMAGES_DIR)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)



In [ ]:
# Cell 4: Model Architecture & Evaluation Helper
def build_resnet18(pretrained=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 1)
    )
    return model

def evaluate_model(model, loader, device):
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images).squeeze(-1)
            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_targets.extend(labels.cpu().numpy().tolist())
            all_probs.extend(probs.tolist())

    avg_loss = total_loss / max(1, len(all_targets))
    auc = float(roc_auc_score(all_targets, all_probs)) if len(np.unique(all_targets)) > 1 else 0.5
    return avg_loss, auc, all_targets, all_probs



In [ ]:
# Cell: Client 1 Fine-Tuning (1 Local Epoch on Dirichlet Partition Index 0)
CLIENT_ID = 1
PARTITION_IDX = 0  # Client 1 in Dirichlet split (1366 samples, 57.0% positive)
LOCAL_EPOCHS = 1
POS_WEIGHT_VAL = 3.2596

pdir = find_client_partitions()
part_file = pdir / f"client_{PARTITION_IDX}.csv"

if not part_file.exists():
    raise FileNotFoundError(
        f"CRITICAL ERROR: Partition file client_{PARTITION_IDX}.csv not found at {part_file}! "
        "Synthetic data fallback is strictly prohibited. Please attach the Step 10 partition dataset to this notebook."
    )

print(f"[CLIENT 1] Found partition file at: {part_file}")
client_df = pd.read_csv(part_file)
pos_count = int(client_df["Target"].sum())
neg_count = len(client_df) - pos_count
pos_pct = client_df["Target"].mean() * 100

print(f"[CLIENT 1] Partition Summary: {len(client_df)} samples | Normal: {neg_count} | Pneumonia: {pos_count} ({pos_pct:.1f}% positive)")

client_dataset = RSNADICOMDataset(client_df, IMAGES_DIR)
client_loader = DataLoader(client_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)

global_checkpoint_path = find_checkpoint("best_fedprox_model_mu_0.1.pt")
if not global_checkpoint_path.exists():
    print("[INFO] Step 11 checkpoint best_fedprox_model_mu_0.1.pt not found. Checking Step 5 baseline checkpoint...")
    global_checkpoint_path = find_checkpoint("best_baseline_model.pt")

if not global_checkpoint_path.exists():
    raise FileNotFoundError("CRITICAL ERROR: No initial global checkpoint (Step 11 or Step 5) found! Cannot begin fine-tuning.")

print(f"[CLIENT 1] Loading initial global weights from: {global_checkpoint_path}")
model = build_resnet18(pretrained=True).to(device)
model.load_state_dict(torch.load(global_checkpoint_path, map_location=device))

model.eval()
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT_VAL], dtype=torch.float32).to(device))
pre_loss = 0.0
with torch.no_grad():
    for images, labels in client_loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images).squeeze(-1)
        loss = criterion(logits, labels)
        pre_loss += loss.item() * len(labels)
pre_loss = pre_loss / max(1, len(client_df))

print(f"[CLIENT 1] Pre-training Initial Local Loss: {pre_loss:.4f}")

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
model.train()
post_loss = 0.0
total_samples = 0

start_time = time.time()
for images, labels in client_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    logits = model(images).squeeze(-1)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    post_loss += loss.item() * len(labels)
    total_samples += len(labels)

post_loss = post_loss / max(1, total_samples)
elapsed = time.time() - start_time

print(f"[CLIENT 1] 1 Local Epoch Complete in {elapsed:.2f}s | Post-training Loss: {post_loss:.4f}")

weights_path = OUTPUT_DIR / "client1_weights.pt"
metadata_path = OUTPUT_DIR / "client1_metadata.json"

torch.save(model.state_dict(), weights_path)

metadata = {
    "client_id": 1,
    "hardware_assignment": "Friend's PC (Ryzen 7, RTX 4050, 16GB RAM)",
    "partition_index": PARTITION_IDX,
    "partition_size": len(client_df),
    "positive_count": pos_count,
    "positive_percentage": round(pos_pct, 2),
    "epochs_trained": LOCAL_EPOCHS,
    "pre_train_loss": round(pre_loss, 4),
    "post_train_loss": round(post_loss, 4),
    "training_time_seconds": round(elapsed, 2),
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"[CLIENT 1] Local training complete. Weights saved: client1_weights.pt")
print(f"[CLIENT 1] Metadata saved: client1_metadata.json")



---

### 💾 Kaggle Output Persistence & Cross-Session Dataset Saving

> [!IMPORTANT]
> Kaggle's `/kaggle/working` directory is **ephemeral** and cleared when a session ends.
> To persist model checkpoints, client partitions, plots, and markdown reports across separate Kaggle sessions without retraining:
> 1. Click **Save Version** (top right menu) $\rightarrow$ Select **Save & Run All (Commit)** $\rightarrow$ Click **Save**.
> 2. Once completed, navigate to your notebook output page $\rightarrow$ Click **Create Dataset** (e.g., name it `federated-medical-ai-outputs`).
> 3. In future sessions (e.g., Step 7 Grad-CAM or Step 11 FedProx): Click **+ Add Input** $\rightarrow$ Search for `federated-medical-ai-outputs` $\rightarrow$ Click **Add**.
> 4. The notebooks will automatically discover `/kaggle/input/federated-medical-ai-outputs/checkpoints/` and load pre-saved weights without retraining!



In [ ]:
# Cell: Kaggle Output Persistence & Dataset Auto-Packager
import zipfile

zip_path = Path("/kaggle/working/outputs_bundle.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in OUTPUT_DIR.rglob("*"):
        if file.is_file() and file.name != "outputs_bundle.zip":
            arcname = file.relative_to(OUTPUT_DIR)
            zipf.write(file, arcname)

print("=" * 85)
print("  KAGGLE OUTPUT PERSISTENCE & CROSS-SESSION REUSE INSTRUCTIONS")
print("=" * 85)
print(f"[OK] Successfully packaged all checkpoints, plots, and reports into: {zip_path}")
print()
print("To reuse this checkpoint / dataset in future Kaggle sessions:")
print(" 1. In top right notebook menu: Click 'Save Version' -> Select 'Save & Run All' -> Save.")
print(" 2. OR go to your notebook output page -> Click 'Create Dataset' -> Name it 'federated-medical-ai-outputs'.")
print(" 3. In future sessions (e.g., Step 7 Grad-CAM or Step 11 FedProx):")
print("    - Click '+ Add Input' in the right sidebar -> Search for 'federated-medical-ai-outputs' -> Click 'Add'.")
print("    - Notebooks will automatically detect pre-saved checkpoints from:")
print("      /kaggle/input/federated-medical-ai-outputs/checkpoints/...")
print("      and reuse real model weights without silently retraining!")
print("=" * 85)

